# English to Hindi Machine Translation

This notebook demonstrates how to build and train a sequence-to-sequence model for English-to-Hindi machine translation using the Hugging Face Transformers library.

In [ ]:
# Install necessary libraries
!pip install -q transformers datasets sentencepiece sacrebleu accelerate evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.9/65.9 kB 7.1 MB/s eta 0:00:00


In [ ]:
import numpy as np
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer
)
import evaluate

## 1. Load Dataset

We will use the `cfilt/iitb-english-hindi` dataset from Hugging Face Datasets.

In [ ]:
dataset = load_dataset("cfilt/iitb-english-hindi")
print(dataset)

README.md:   0%|          | 0.00/3.14k [00:00<?, ?B/s]

dataset_infos.json:   0%|          | 0.00/953 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  190MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 85.7kB            

...

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/1659083 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/520 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2507 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['translation'],
        num_rows: 1659083
    })
    validation: Dataset({
        features: ['translation'],
        num_rows: 520
    })
    test: Dataset({
        features: ['translation'],
        num_rows: 2507
    })
})


### Split and Sample Data

To speed up training for demonstration, we'll sample a smaller subset of the dataset.

In [ ]:
train_dataset = dataset["train"].shuffle(seed=42).select(range(5000))
validation_dataset = dataset["validation"].select(range(500))
test_dataset = dataset["test"].select(range(500))

print(f"Train dataset size: {len(train_dataset)}")
print(f"Validation dataset size: {len(validation_dataset)}")
print(f"Test dataset size: {len(test_dataset)}")

Train dataset size: 5000
Validation dataset size: 500
Test dataset size: 500


## 2. Initialize Tokenizer and Model

We will use the `google/mt5-base` model and its tokenizer, which is a multilingual T5 model suitable for translation tasks.

In [ ]:
model_name = "google/mt5-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

config.json:   0%|          | 0.00/702 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/376 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/65.0 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 2.33GB            

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.33GB            

Loading weights:   0%|          | 0/284 [00:00<?, ?it/s]

model.safetensors: downloading bytes:           |  0.00B            

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

## 3. Preprocess Data

We need to tokenize the English and Hindi sentences and prepare them for the model.

In [ ]:
max_input_length = 64
max_target_length = 64

def preprocess(example):
    source = example["translation"]["en"]
    target = example["translation"]["hi"]

    inputs = tokenizer(
        source,
        max_length=max_input_length,
        truncation=True
    )

    labels = tokenizer(
        text_target=target,
        max_length=max_target_length,
        truncation=True
    )

    inputs["labels"] = labels["input_ids"]

    return inputs

In [ ]:
tokenized_train = train_dataset.map(
    preprocess,
    remove_columns=train_dataset.column_names
)

tokenized_validation = validation_dataset.map(
    preprocess,
    remove_columns=validation_dataset.column_names
)

tokenized_test = test_dataset.map(
    preprocess,
    remove_columns=test_dataset.column_names
)

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

## 4. Data Collator

`DataCollatorForSeq2Seq` will handle padding for batches.

In [ ]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model
)

## 5. Metrics Definition

We will use BLEU score to evaluate the translation quality.

In [ ]:
bleu = evaluate.load("sacrebleu")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred

    # Handle potential NaN or out-of-range values in predictions
    # Replace NaNs with pad_token_id, then convert to int64 for safe decoding
    if np.isnan(predictions).any():
        predictions = np.nan_to_num(predictions, nan=tokenizer.pad_token_id)

    # Ensure predictions are integer type. Using int64 for robustness.
    predictions = predictions.astype(np.int64)
    # Replace any negative values (like -100 if present) with pad_token_id
    predictions = np.where(predictions < 0, tokenizer.pad_token_id, predictions)

    decoded_preds = tokenizer.batch_decode(
        predictions,
        skip_special_tokens=True
    )

    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    decoded_labels = tokenizer.batch_decode(
        labels,
        skip_special_tokens=True
    )

    decoded_labels = [[label] for label in decoded_labels] # sacrebleu expects list of references

    result = bleu.compute(
        predictions=decoded_preds,
        references=decoded_labels
    )

    return {"BLEU": result["score"]}

## 6. Training Arguments and Trainer

Define the training arguments and instantiate the `Seq2SeqTrainer`.

In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir="./translation_model",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    weight_decay=0.01,
    save_total_limit=2,
    num_train_epochs=2,
    predict_with_generate=True,
    logging_steps=100,
    fp16=False # Set to False to avoid numerical instability issues encountered previously
)

In [ ]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_validation,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)
trainer.train()

Epoch,Training Loss,Validation Loss,Bleu
1,5.356246,3.460165,0.068109
2,4.685938,3.087106,0.175722


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/transformers/generation/utils.py:1676: UserWarning: Using the model-agnostic default `max_length` (=21) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1250, training_loss=7.066655554199219, metrics={'train_runtime': 1435.0048, 'train_samples_per_second': 6.969, 'train_steps_per_second': 0.871, 'total_flos': 1124368039821312.0, 'train_loss': 7.066655554199219, 'epoch': 2.0})

## 7. Evaluate the Model

Evaluate the trained model on the validation set.

In [ ]:
results = trainer.evaluate()
print(results)

Training Loss,Validation Loss,Epoch,Bleu
4.685938,3.087106,2,0.175722


{'eval_loss': 3.0871057510375977, 'eval_BLEU': 0.17572185943735477}


## 8. Translate Function

Define a function to translate an English sentence to Hindi.

In [ ]:
# Get the device of the model (assuming it's on CUDA if available)
model_device = model.device

def translate(sentence):
    inputs = tokenizer(
        sentence,
        return_tensors="pt"
    ).to(model_device) # Move inputs to the model's device

    outputs = model.generate(
        **inputs,
        max_length=max_target_length # Use the predefined max_target_length
    )

    return tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

### Test Translation

In [ ]:
sentence = "Artificial Intelligence is changing the world."

print("English:")
print(sentence)

print()

print("Hindi:")
print(translate(sentence))

English:
Artificial Intelligence is changing the world.

Hindi:
<extra_id_0> और नई चीजों के लिए प्रयोग करने के लिए प्रयोग करने के लिए प्रयोग करने के लिए प्रयोग करने के लिए प्रयोग करने के लिए प्रयोग करने के लिए प्रयोग करने के लिए 


## 9. Save Model and Tokenizer

Save the fine-tuned model and tokenizer for future use.

In [ ]:
model.save_pretrained("./english_hindi_transformer")
tokenizer.save_pretrained("./english_hindi_transformer")
print("Model and tokenizer saved to ./english_hindi_transformer")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model and tokenizer saved to ./english_hindi_transformer
